In [ ]:
import dynamiqs as dq
dq.set_precision('double')
import numpy as np
from scipy.sparse.linalg import eigs
import scipy.sparse as sp
import scipy.sparse.linalg as spla

import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
kappa_b = 0.03
E_J = 37*2*np.pi /2
phi_a = 0.11
phi_b = 0.204
epsilon_p = 0.02
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2

In [ ]:
rates_test = []
alpha_sq_list = np.linspace(1,4,10)

# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
E_J = 37*2*np.pi /2
phi_a = 0.11
phi_b = 0.204
epsilon_p = 0.02
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
kappa_2 = 4*g**2/kappa_b
kappa_1 = 1e-6
n_a = 25
n_b = 13
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

# Correct nonlinear term
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)
drive_op = b_tot + b_tot.dag()

La = np.sqrt(kappa_1) * dq.dag(a_tot)
Lb = np.sqrt(kappa_b) * b_tot

# Convert to sparse CSC
La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
I = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )
rates_list_arnoldi = []
for alpha_sq in alpha_sq_list:
    print(alpha_sq)
    epsilon_d = 2*alpha_sq*g
    H = g * non_linear_op + (epsilon_d / 2) * drive_op
    H = sp.csc_matrix(H.to_numpy().astype(np.complex128))
    
    Lio = (
    -1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
    + dissipator(La)
    + dissipator(Lb)
    )

    vals, vecs = spla.eigs(Lio, k=6, sigma=1e-12, which="LM", tol=1e-10)
    rates_list_arnoldi.append(-np.sort(vals)[-2].real)
rates_test.append(rates_list_arnoldi)